# 🌐 Notebook 5: Distributed Coordination

When data spans multiple databases, we need **distributed coordination**. This is significantly more complex than single-database solutions and should be avoided when possible.

## Learning Objectives

By the end of this notebook, you'll understand:
- Two-Phase Commit (2PC) and its limitations
- Distributed locks with Redis
- The Saga pattern for resilient distributed transactions
- When to use each approach

## 🎯 When Do You Need Distributed Coordination?

**Before reaching for distributed coordination, ask:**

1. Can I keep all the data in ONE database? (usually yes!)
2. Can I use eventual consistency instead of strong consistency?
3. Is the complexity worth it?

**You need distributed coordination when:**
- Data is sharded across multiple databases
- You need atomicity across different services
- Consistency requirements prevent eventual consistency

---

### 🔍 Open RedisInsight to Watch Distributed Locks

1. Go to **http://localhost:5540**
2. Click "Add Redis Database"
3. Enter: Host = `redis`, Port = `6379`, Name = `contention-redis`
4. Click "Add Redis Database"
5. Open the "Browser" tab to see keys

💡 **Tip**: Watch for `lock:*` keys appearing when you run the distributed lock demos!

In [ ]:
import psycopg2
import redis
import time
import uuid
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "contention_demo",
    "user": "demo",
    "password": "demo"
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_connection():
    return redis.Redis(host='localhost', port=6379, decode_responses=True)

event_log = []
log_lock = Lock()

def log_event(user: str, event: str):
    with log_lock:
        event_log.append({"time": time.time(), "user": user, "event": event})

try:
    conn = get_db_connection()
    print("✅ PostgreSQL connected")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_connection()
    r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Make sure Redis is running: docker-compose up -d")

## 🔒 Distributed Locks with Redis

Distributed locks coordinate access to resources across multiple application instances. Redis is popular for this because it's fast and atomic.

In [ ]:
class RedisLock:
    def __init__(self, redis_client, lock_name: str, ttl_seconds: int = 30):
        self.redis = redis_client
        self.lock_name = f"lock:{lock_name}"
        self.ttl = ttl_seconds
        self.token = None
    
    def acquire(self, timeout: float = 10) -> bool:
        self.token = str(uuid.uuid4())
        start_time = time.time()
        
        while time.time() - start_time < timeout:
            if self.redis.set(self.lock_name, self.token, nx=True, ex=self.ttl):
                return True
            time.sleep(0.01)
        
        return False
    
    def release(self) -> bool:
        if not self.token:
            return False
        
        script = """
        if redis.call('get', KEYS[1]) == ARGV[1] then
            return redis.call('del', KEYS[1])
        else
            return 0
        end
        """
        
        result = self.redis.eval(script, 1, self.lock_name, self.token)
        self.token = None
        return result == 1
    
    def __enter__(self):
        if not self.acquire():
            raise Exception(f"Failed to acquire lock: {self.lock_name}")
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.release()

print("✅ RedisLock class created")

In [ ]:
def reset_concert(seats: int = 1):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE concerts SET available_seats = %s WHERE id = 1", (seats,))
    cursor.execute("DELETE FROM tickets WHERE concert_id = 1")
    conn.commit()
    conn.close()

def buy_ticket_with_redis_lock(user_id: str) -> dict:
    r = get_redis_connection()
    lock = RedisLock(r, "concert:1:purchase", ttl_seconds=10)
    
    log_event(user_id, "Trying to acquire lock...")
    
    if not lock.acquire(timeout=5):
        log_event(user_id, "❌ Failed to acquire lock")
        return {"success": False, "reason": "lock_timeout"}
    
    log_event(user_id, "🔒 Lock acquired!")
    
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        
        cursor.execute("SELECT available_seats FROM concerts WHERE id = 1")
        seats = cursor.fetchone()[0]
        
        time.sleep(0.1)
        
        if seats >= 1:
            cursor.execute(
                "UPDATE concerts SET available_seats = available_seats - 1 WHERE id = 1"
            )
            cursor.execute(
                "INSERT INTO tickets (concert_id, user_id, seat_number, purchase_price) "
                "VALUES (1, %s, 'A1', 150.00)",
                (user_id,)
            )
            conn.commit()
            log_event(user_id, "✅ Purchase successful")
            return {"success": True, "user": user_id}
        else:
            log_event(user_id, "❌ Sold out")
            return {"success": False, "reason": "sold_out"}
            
    finally:
        lock.release()
        log_event(user_id, "🔓 Lock released")
        conn.close()

print("🔒 Testing Distributed Lock with Redis")
print("=" * 50)

reset_concert(1)
event_log.clear()

with ThreadPoolExecutor(max_workers=2) as executor:
    f1 = executor.submit(buy_ticket_with_redis_lock, "Alice")
    f2 = executor.submit(buy_ticket_with_redis_lock, "Bob")
    
    r1 = f1.result()
    r2 = f2.result()

print("\n📋 Event Log:")
for event in sorted(event_log, key=lambda x: x["time"]):
    print(f"   [{event['user']:5s}] {event['event']}")

print(f"\n📊 Results:")
print(f"   Alice: {r1}")
print(f"   Bob:   {r2}")

## 🎫 Seat Reservations Pattern

A better UX pattern: **reserve seats temporarily** instead of competing for purchase locks.

In [ ]:
from datetime import datetime, timedelta

def reset_seats():
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE seat_reservations SET status = 'available', "
        "user_id = NULL, reserved_until = NULL WHERE concert_id = 1"
    )
    conn.commit()
    conn.close()

def reserve_seat(user_id: str, seat_number: str, reservation_minutes: int = 10) -> dict:
    conn = get_db_connection()
    cursor = conn.cursor()
    
    try:
        reserved_until = datetime.now() + timedelta(minutes=reservation_minutes)
        
        cursor.execute(
            "UPDATE seat_reservations "
            "SET status = 'reserved', user_id = %s, reserved_until = %s "
            "WHERE concert_id = 1 AND seat_number = %s "
            "AND (status = 'available' OR (status = 'reserved' AND reserved_until < NOW()))",
            (user_id, reserved_until, seat_number)
        )
        
        if cursor.rowcount == 0:
            conn.rollback()
            return {"success": False, "reason": "seat_unavailable"}
        
        conn.commit()
        return {
            "success": True, 
            "user": user_id, 
            "seat": seat_number,
            "expires": reserved_until.isoformat()
        }
        
    except Exception as e:
        conn.rollback()
        return {"success": False, "reason": str(e)}
    finally:
        conn.close()

def complete_purchase(user_id: str, seat_number: str) -> dict:
    conn = get_db_connection()
    cursor = conn.cursor()
    
    try:
        cursor.execute(
            "UPDATE seat_reservations "
            "SET status = 'sold' "
            "WHERE concert_id = 1 AND seat_number = %s "
            "AND user_id = %s AND status = 'reserved' AND reserved_until > NOW()",
            (seat_number, user_id)
        )
        
        if cursor.rowcount == 0:
            conn.rollback()
            return {"success": False, "reason": "reservation_expired_or_invalid"}
        
        conn.commit()
        return {"success": True, "user": user_id, "seat": seat_number}
        
    except Exception as e:
        conn.rollback()
        return {"success": False, "reason": str(e)}
    finally:
        conn.close()

print("🎫 Seat Reservation Pattern")
print("=" * 50)

reset_seats()

print("\nAlice and Bob both try to reserve seat A1...")

with ThreadPoolExecutor(max_workers=2) as executor:
    f1 = executor.submit(reserve_seat, "Alice", "A1")
    f2 = executor.submit(reserve_seat, "Bob", "A1")
    
    r1 = f1.result()
    r2 = f2.result()

print(f"\nAlice: {r1}")
print(f"Bob:   {r2}")

winner = "Alice" if r1["success"] else "Bob"
print(f"\n{winner} got the reservation! They have 10 minutes to complete payment.")

print("\n💡 Benefits:")
print("   - Users don't compete during checkout")
print("   - Failed payments auto-release seats (TTL)")
print("   - Better UX than 'seat taken' at payment")

## 📜 The Saga Pattern

Sagas break distributed transactions into **independent steps** with **compensating actions** for rollback.

In [ ]:
import json

class TransferSaga:
    def __init__(self, from_user: str, to_user: str, amount: float):
        self.transaction_id = str(uuid.uuid4())[:8]
        self.from_user = from_user
        self.to_user = to_user
        self.amount = amount
        self.steps_completed = []
    
    def log_step(self, step_name: str, status: str, data: dict = None):
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO transaction_log (transaction_id, step_name, status, data) "
            "VALUES (%s, %s, %s, %s)",
            (self.transaction_id, step_name, status, json.dumps(data or {}))
        )
        conn.commit()
        conn.close()
    
    def debit_from_account(self) -> bool:
        conn = get_db_connection()
        cursor = conn.cursor()
        
        try:
            cursor.execute(
                "UPDATE accounts SET balance = balance - %s "
                "WHERE user_id = %s AND balance >= %s",
                (self.amount, self.from_user, self.amount)
            )
            
            if cursor.rowcount == 0:
                conn.rollback()
                self.log_step("debit", "failed", {"reason": "insufficient_funds"})
                return False
            
            conn.commit()
            self.steps_completed.append("debit")
            self.log_step("debit", "completed", {"amount": self.amount})
            return True
            
        except Exception as e:
            conn.rollback()
            self.log_step("debit", "failed", {"error": str(e)})
            return False
        finally:
            conn.close()
    
    def credit_to_account(self) -> bool:
        conn = get_db_connection()
        cursor = conn.cursor()
        
        try:
            cursor.execute(
                "UPDATE accounts SET balance = balance + %s WHERE user_id = %s",
                (self.amount, self.to_user)
            )
            
            if cursor.rowcount == 0:
                conn.rollback()
                self.log_step("credit", "failed", {"reason": "account_not_found"})
                return False
            
            conn.commit()
            self.steps_completed.append("credit")
            self.log_step("credit", "completed", {"amount": self.amount})
            return True
            
        except Exception as e:
            conn.rollback()
            self.log_step("credit", "failed", {"error": str(e)})
            return False
        finally:
            conn.close()
    
    def compensate_debit(self):
        conn = get_db_connection()
        cursor = conn.cursor()
        
        try:
            cursor.execute(
                "UPDATE accounts SET balance = balance + %s WHERE user_id = %s",
                (self.amount, self.from_user)
            )
            conn.commit()
            self.log_step("compensate_debit", "completed", {"amount": self.amount})
            print(f"   💰 Compensated: Credited ${self.amount} back to {self.from_user}")
        finally:
            conn.close()
    
    def execute(self) -> dict:
        print(f"\n📜 Saga {self.transaction_id}: Transfer ${self.amount} from {self.from_user} to {self.to_user}")
        print("─" * 50)
        
        print(f"   Step 1: Debit {self.from_user}...")
        if not self.debit_from_account():
            return {"success": False, "reason": "debit_failed"}
        print(f"   ✅ Debited ${self.amount} from {self.from_user}")
        
        print(f"   Step 2: Credit {self.to_user}...")
        if not self.credit_to_account():
            print(f"   ❌ Credit failed! Running compensation...")
            self.compensate_debit()
            return {"success": False, "reason": "credit_failed_compensated"}
        print(f"   ✅ Credited ${self.amount} to {self.to_user}")
        
        print(f"   🎉 Saga completed successfully!")
        return {"success": True, "transaction_id": self.transaction_id}

print("✅ TransferSaga class created")

In [ ]:
def reset_balances():
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE accounts SET balance = 1000 WHERE user_id = 'alice'")
    cursor.execute("UPDATE accounts SET balance = 500 WHERE user_id = 'bob'")
    cursor.execute("DELETE FROM transaction_log")
    conn.commit()
    conn.close()

def show_balances():
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT user_id, balance FROM accounts WHERE user_id IN ('alice', 'bob') ORDER BY user_id")
    accounts = cursor.fetchall()
    conn.close()
    
    total = 0
    for user_id, balance in accounts:
        print(f"   {user_id}: ${balance:.2f}")
        total += float(balance)
    print(f"   Total: ${total:.2f}")

print("📜 Saga Pattern Demo - Successful Transfer")
print("=" * 50)

reset_balances()
print("\nBefore:")
show_balances()

saga = TransferSaga("alice", "bob", 100)
result = saga.execute()

print("\nAfter:")
show_balances()

In [ ]:
print("📜 Saga Pattern Demo - Failed Transfer with Compensation")
print("=" * 50)

reset_balances()
print("\nBefore:")
show_balances()

print("\nTransferring to non-existent user 'charlie_fake'...")

saga = TransferSaga("alice", "charlie_fake", 100)
result = saga.execute()

print("\nAfter (should be unchanged due to compensation):")
show_balances()

print("\n💡 The saga debited Alice, failed to credit, then compensated!")

## ⚖️ Choosing the Right Approach

| Approach | Use When | Complexity | Consistency |
|----------|----------|------------|-------------|
| **Single DB Transaction** | Data in one DB | Low | Strong |
| **Distributed Lock** | Coordination across services | Medium | Strong |
| **Saga** | Long-running distributed ops | High | Eventual |
| **2PC** | Must have atomicity across DBs | Very High | Strong |

In [ ]:
print("📊 Decision Guide")
print("=" * 60)
print()
print("Question 1: Is all your data in ONE database?")
print("   YES → Use database transactions + locks")
print("   NO  → Continue to question 2")
print()
print("Question 2: Can you tolerate brief inconsistency?")
print("   YES → Use Saga pattern")
print("   NO  → Continue to question 3")
print()
print("Question 3: Do you need to coordinate just one operation?")
print("   YES → Use Distributed Lock (Redis)")
print("   NO  → Consider 2PC (but really try to avoid it!)")
print()
print("💡 90% of the time, you can keep data in one DB!")

## 🧪 Quick Quiz

1. **Why use Redis for distributed locks instead of the database?**

2. **What's a compensating action in the Saga pattern?**

3. **Why should you avoid 2PC when possible?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Redis for distributed locks:")
print("   - Faster than database locks")
print("   - Built-in TTL for automatic cleanup")
print("   - Works across multiple application instances")
print("   - Database locks are per-connection")
print()
print("2. Compensating action:")
print("   - An operation that 'undoes' a previous step")
print("   - Example: Credit back money after failed transfer")
print("   - Runs when later steps fail")
print("   - Must be idempotent!")
print()
print("3. Why avoid 2PC:")
print("   - Coordinator crash leaves participants uncertain")
print("   - Holds locks across network calls (dangerous!)")
print("   - Very complex to implement correctly")
print("   - Single point of failure")

## 📚 Summary

### What We Learned

1. **Try to avoid distributed coordination** - keep data in one DB!
2. **Distributed locks** (Redis) - for simple cross-service coordination
3. **Reservations pattern** - better UX than competing for locks
4. **Saga pattern** - for resilient distributed transactions
5. **2PC** - avoid if possible, use sagas instead

### Key Interview Points

> "Before reaching for distributed coordination, I'd first try to keep all contended data in a single database. If that's not possible, I'd use the Saga pattern for resilience, with distributed locks for simpler coordination needs."

### Pattern Summary

| Problem | Single DB | Multi DB |
|---------|-----------|----------|
| High contention | Pessimistic locking | Distributed lock |
| Low contention | Optimistic concurrency | Saga + retry |
| User-facing | Reservations | Reservations + distributed lock |
| Must be atomic | Transaction | Saga (eventual) or 2PC (avoid) |